In [1]:
import os
import xml.etree.ElementTree as ET

import pandas as pd

In [2]:
def is_img(pth):
    return os.path.splitext(pth)[-1].lower() in [".png", ".jpg", ".jpeg", ".bmp"]

In [4]:
def parse_text(txt):
    if txt == "true":
        return True
    elif txt == "false":
        return False
    else:
        return txt

def split_point(s):
    return s.split(";")[0].split(",")

def process_img_node(node: ET.Element) -> list:
    img_attributes = {
        "name": node.attrib["name"],
        "width": int(node.attrib["width"]),
        "height": int(node.attrib["height"])
    }

    boxes = []
    positions = []
    for child in node:
        attributes = {}
        for meta in child:
            attributes[meta.attrib["name"]] = parse_text(meta.text)

        obj = {
            **img_attributes,
            "label": child.attrib["label"],
            **attributes
        }
        if child.tag == "points":
            obj["pos"] = [float(i) for i in split_point(child.attrib["points"])]
            positions.append(obj)
        else:
            obj["box"] = [
                float(child.attrib["xtl"]),
                float(child.attrib["ytl"]),
                float(child.attrib["xbr"]),
                float(child.attrib["ybr"])
            ]
            boxes.append(obj)

    if len(boxes) != len(positions):
        print(f"[WARNING] Number of boxes not equal to number of positions: {len(boxes)} vs {len(positions)}")

    objects = []
    for box in boxes:
        for pos in positions:
            if pos["ID"] == box["ID"]:
                box["pos"] = pos["pos"]
                objects.append(box)

    return objects

def process_annotation_file(pth: str, batch: str = None, folder: str = None) -> list:
    tree = ET.parse(pth)
    root = tree.getroot()
    objects = []
    for child in root:
        if child.tag != "image":
            continue
        objects += process_img_node(child)

    if batch is not None:
        for obj in objects:
            obj["batch"] = batch

    if folder is not None:
        for obj in objects:
            obj["folder"] = folder

    return objects

In [9]:
base_path = "../../../db/object-detection"
batches = sorted([i for i in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, i))])

folders = {}
for batch in batches:
    tmp_pth = os.path.join(base_path, batch)
    folders[batch] = sorted([i for i in os.listdir(tmp_pth) if os.path.isdir(os.path.join(tmp_pth, i))])

In [19]:
imgs = {}
annotations = {}
total_img_count = 0
for batch in batches:
    for folder in folders[batch]:
        tmp_pth = os.path.join(base_path, batch, folder)
        batch_imgs = [i for i in os.listdir(tmp_pth) if is_img(i)]
        batch_annotation = [i for i in os.listdir(tmp_pth) if i.endswith(".xml")][0]
        
        key = (batch, folder)
        imgs[key] = batch_imgs
        annotations[key] = batch_annotation
        total_img_count += len(batch_imgs)

print("total images:", total_img_count)

total images: 609


In [36]:
objects = []

for batch, annot in annotations.items():
    objects += process_annotation_file(os.path.join(base_path, *batch, annot), *batch)

df = pd.DataFrame(objects)

In [37]:
print("total number of images: ", df["name"].unique().shape[0])

print("total number of annotated objects: ", df.shape[0])

object_per_img = df.groupby("name").count()["width"].values
print(f"objects per img: min = {object_per_img.min()}. max = {object_per_img.max()}. avg = {object_per_img.mean():.02f}")

print("number of object per class:\n", df["label"].value_counts())

total number of images:  599
total number of annotated objects:  13424
objects per img: min = 1. max = 81. avg = 22.41
number of object per class:
 label
office_chair               4099
desk                       1617
desk_FULL                  1263
laptop_opened               861
cabinet_FULL                860
cabinet                     750
monitor                     657
dinning_chair               475
bag                         383
laptop_closed               327
coffee_table                316
meeting_room_table          295
plants                      261
coffee_table_FULL           215
keyboard                    208
lounge_chair                194
whiteboard                  190
coat                        146
bar_stool                   146
sofa                         68
meeting_room_table_FULL      59
printer                      20
coffee_station                8
sink                          6
Name: count, dtype: int64


In [76]:
if df["name"].unique().shape[0] == total_img_count:
    print("OK")
elif df["name"].unique().shape[0] > total_img_count:
    tmp = []
    for i in imgs.values():
        tmp += i

    not_uploaded_imgs = set(df["name"].unique()).difference(tmp)
    print("total number of not uploaded images:", len(not_uploaded_imgs))
    print("batch | image")
    print("-" * 25)
    for i in not_uploaded_imgs:
        tmp = df[df["name"] == i][["name", "batch"]].iloc[0]
        print(f"{tmp['batch']:>5} | {tmp['name']}")
else:
    print("annotated images")
    print("batch, folder             | image")
    print("-" * 25)
    for k, v in imgs.items():
        tmp = set(v).difference(df["name"].unique())
        for i in tmp:
            print(f"{str(k):<25} | {i}")
            
    print()
    print("duplicates")
    df1 = df.groupby(['name','batch', "folder"]).size().reset_index()[['name','batch', "folder"]]

    duplicates = df1[df1.duplicated('name', keep=False)]

    duplicate_info = duplicates.groupby('name').agg({
        'folder': lambda x: ', '.join(x),
        'batch': lambda x: ', '.join(x)
    }).reset_index()
    
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)

    print(duplicate_info)

annotated images
batch, folder             | image
-------------------------
('batch3_21.5', '12')     | PG1_DL_H250_Walk_Beige_Carpet_17_09_34_45_2016-5_266.jpg

duplicates
                                                      name folder                     batch
0      Cam3_H230_spacesks_os1_12024-03-03-20h06m52s241.png  10, 4  batch2_20.5, batch3_21.5
1      Cam3_H230_spacesks_os1_12024-03-03-20h07m06s021.png  10, 4  batch2_20.5, batch3_21.5
2      Cam3_H230_spacesks_os1_12024-03-03-20h07m12s958.png  10, 4  batch2_20.5, batch3_21.5
3      Cam3_H230_spacesks_os1_12024-03-03-20h07m19s520.png  10, 4  batch2_20.5, batch3_21.5
4    Cam3_H235_spaceKS_seats_1_2024-03-04-11h17m28s042.png  10, 4  batch2_20.5, batch3_21.5
5    Cam3_H235_spaceKS_seats_1_2024-03-04-11h17m35s301.png  10, 4  batch2_20.5, batch3_21.5
6  Cam3_H240_spaceKS_objects_1_2024-03-04-09h57m48s039.png  10, 4  batch2_20.5, batch3_21.5
7  Cam3_H240_spaceKS_objects_1_2024-03-04-09h57m51s988.png  10, 4  batch2_20.5, batch3_21.

In [77]:
df.to_csv(os.path.join(base_path, "annotation_all.csv"), index=False)